# Task 5 — Source Metadata Ingestion into MongoDB

## Approach and reasoning

A Spark Structured Streaming job consumes `cpg.metadata` and writes through the
MongoDB Spark Connector into `cpg.file_metadata`.

Two design decisions carry the lab's requirements:

**1. Checkpointing.** `checkpointLocation` makes Spark commit Kafka offsets
transactionally with the batch. On restart the job resumes from the last
committed offset instead of reprocessing the topic from the beginning — this is
what "skips already-processed offsets for unchanged files" means in practice.

**2. Upsert instead of append.** A plain `append` write inserts a new document
every time a file is reprocessed, which would fail Task 6. Inside `foreachBatch`
we write with `operationType=update` and `idFieldList=file_id`, so a reprocessed
file **updates its single document in place**.

`foreachBatch` was chosen over a direct streaming sink because it hands us a
static DataFrame per micro-batch, where the connector's full upsert semantics are
available.

### Step 1 — start the streaming job

Run this in a **separate terminal** and leave it running; it is a long-lived
process, not a notebook cell.

```bash
spark-submit \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.mongodb.spark:mongo-spark-connector_2.12:10.4.0 \
  src/spark/spark_mongo_stream.py \
  --bootstrap localhost:9092 \
  --mongo-uri mongodb://localhost:27017 \
  --checkpoint /tmp/chk/cpg_metadata
```

Paste the batch log output below as evidence:

```
batch 0: upserted 59 metadata docs
```

### Step 2 — verify the documents landed in MongoDB

In [ ]:
import os
os.chdir("..")
!docker exec mongodb mongosh --quiet cpg --eval "db.file_metadata.countDocuments()"

### Step 3 — inspect one document

In [ ]:
!docker exec mongodb mongosh --quiet cpg --eval \
  "printjson(db.file_metadata.findOne({rel_path: 'optimum/version.py'}))"

### Step 4 — the duplicate check that must return an empty array

In [ ]:
!docker exec mongodb mongosh --quiet cpg --eval \
  "printjson(db.file_metadata.aggregate([{\$group:{_id:'\$file_id',c:{\$sum:1}}},{\$match:{c:{\$gt:1}}}]).toArray())"

### Step 5 — inspect the checkpoint directory Spark maintains

In [ ]:
!ls -R /tmp/chk/cpg_metadata 2>/dev/null | head -20
!cat /tmp/chk/cpg_metadata/offsets/0 2>/dev/null | tail -1

### Database UI evidence

Connect MongoDB Compass to `mongodb://localhost:27017` and open
`cpg.file_metadata`.

**Insert your screenshots here:**

![MongoDB Compass collection](images/mongo_collection.png)

![Spark UI streaming query](images/spark_ui.png)

## Reflection

*(Replace this with what actually happened on your machine.)*

**What failed.** Our first version used `.mode("append")` on the streaming write.
It worked perfectly on the first pass and silently duplicated every document on
replay — the failure only appeared once we reached Task 6. Moving to
`foreachBatch` with `operationType=update` fixed it.

**What worked.** Declaring an explicit `StructType` for the JSON instead of
letting Spark infer it. Schema inference is unavailable on streaming sources
anyway, and the explicit schema doubles as executable documentation of the
message contract from Task 3.

**Version pinning.** The `--packages` coordinates must match the Spark line and
Scala version exactly. A mismatch produces a `ClassNotFoundException` at submit
time rather than a clear error message.